## Computational Graph development for HLAN

### Load necessary libaries and data

In [2]:
### Import libraries
import pandas as pd
from gensim.models import Word2Vec
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from model_helper import *

In [3]:
### Load data
word_model_path = ## Path to word-emb model
code_model_path = ## Path to code-emb model
x_path = ## Path to train_word-emb 
y_path = ## Path to train_code-emb 
word_model = Word2Vec.load(word_model_path)
code_model = Word2Vec.load(code_model_path)
x = torch.load(x_path, weights_only = True)
y = torch.load(y_path, weights_only = True)

### Perform Initialization for embedding layer and W Projection matrix
Additionally, define hyperparameters and model architecture for graph

In [25]:
### Define hidden size and embedding dim
hidden_size = 100
embed_dim = 100

### Define dimension and padding for the embedding layer loaded from pre-trained model
dim = 100
pad = 0
padded_vec = np.zeros(dim)
padding_token = "<PAD>"

### Dropout prob
dropout_prob = 0.2

In [5]:
#### Initialize weight tensors from the pre-trained code and word Word2Vec models
### Words:
word_to_index, word_weight_tensor = get_word_index_wgts(word_model, padding_token, dim, padded_vector = padded_vec)

### Codes:
W_projection = torch.tensor(code_model.wv.vectors) ## [num_labels, hidden_size * 4]

### First Pass:
1. Example Data
2. Embedding
3. Bi-GRU
4. Word-Level Attenion

In [6]:
### Setup Example data
batch_size = 5
x_comp = x[:batch_size,:,:]
print(x_comp.shape)

torch.Size([5, 25, 100])


In [9]:
## Initialize embedding layer from pre-trained weight tensor
embed_layer = nn.Embedding.from_pretrained(word_weight_tensor)

## Initialize First Bi-GRU Layer (Word Level)
gru_w = nn.GRU(input_size = embed_dim, hidden_size = hidden_size, bidirectional = True, batch_first = True)

### Setup label word level attention ###
W_w = nn.Linear(hidden_size*2,hidden_size*2)
attn_tanh = nn.Tanh()
V_w_l = nn.Parameter(torch.randn(50,hidden_size*2))

### First Pass, Embedding and GRU
embed_comp = embed_layer(x_comp)## [batch_size, number sentences, number words (seq len), embed dim (100)]
embed_comp_reshape = embed_comp.view(-1, embed_comp.shape[2],embed_comp.shape[3])## [batch_size *num_sentences, seq_len, embed_dim ]
hidden_state, hnn = gru_w(embed_comp_reshape)## [batch_size * num_sentences, seq_len, embed_dim]

### First pass, label word level attention representation ###

## get v from eq (4)
hidden_state_reshape = hidden_state.reshape(-1,hidden_state.size(-1));##[batch_size * num_sentences * seq_len, hidden_size]
hidden_rep_step = attn_tanh(W_w(hidden_state_reshape))## [batch_size * num_sentences * seq_len, hidden_size]
v = hidden_rep_step.reshape(-1, hidden_state.size(1),hidden_state.size(-1))## [batch_size * num_sentences,num_words,hidden_size]

## get word level attention eq (4)
a_w_l = torch.matmul(v,V_w_l.T).view(-1,v.size(0),v.size(1))## [label_size, batch_size * num_sent, hidden_size]
a_w_l = F.softmax(a_w_l, dim = 2).unsqueeze(3)## [label_size, batch_size * num_sent, num words, hidden_size * 2]
C_s_l = a_w_l*hidden_state## [label_size, batch_size * num_sent, num_words, hidden_size * 2]
C_s_l = torch.sum(C_s_l,dim = 2)## [label_size, batch_size * num_sent, hidden_size * 2]
print(C_s_l.shape)

torch.Size([50, 125, 200])


Second Pass:
1. Initialize necessary layers
2. Bi-GRU
3. Sentence Level Attention
4. Document Representation
5. Final Logits and W Projection

In [27]:
### Initialize sentence Bi-GRU Layer ###
gru_s = nn.GRU(input_size = hidden_size * 2, hidden_size = hidden_size * 2, bidirectional = True, batch_first = True)

### Initialize W_s and context vector V_s_l
W_s = nn.Linear(hidden_size*4,hidden_size*2) ## W_w_attention_sentence
V_s_l = nn.Parameter(torch.randn(50,hidden_size*2)) ## context_vector_sentence_per_label

### Initialize dropout layer
drop_layer = nn.Dropout(p = dropout_prob)

In [24]:
### first pass, label sentence level attention GRU encoder
C_s_l_reshape = C_s_l.permute(1, 0, 2)## [batch_size * num_sent, num_labels, hidden_size * 2]
S_l, hnn = gru_s(C_s_l_reshape) ## [batch_size * num_sent, num_labels, hidden_size * 4]

### Next step, nonlinear transformation
hidden_rep_step = attn_tanh(W_s(S_l)) ## [batch_size * num_sent, num_labels, hidden_size * 2]
U = hidden_rep_step.reshape(hidden_rep_step.size(1), batch_size, -1, hidden_rep_step.size(-1))## [num_labels, batch_size, num_sentences, hidden_size * 2]

### Get Sentence level attention and document representation
S_l_reshape = S_l.reshape(S_l.size(1), batch_size, -1, S_l.size(2))## [num_labels, batch_size, num_sentences, hidden_size * 4]
V_s_l_expand = V_s_l.unsqueeze(1).unsqueeze(1)## [num_labels, 1, 1, hidden_size * 2]
attention_logits = (U * V_s_l_expand).sum(dim=3)## [num_labels, batch_size, num_sentences]
p_attention_sent = F.softmax(attention_logits - attention_logits.max(dim=2, keepdim=True).values, dim=2)## [num_labels, batch_size, hidden_size * 4]
document_representation = (p_attention_sent.unsqueeze(3) * S_l_reshape).sum(dim=2); print(document_representation.shape)## [num_labels, batch_size, hidden_size * 4]

torch.Size([50, 5, 400])


In [31]:
### Final steps for dropout layer and using pre-trained label dimension for final projection
drop_step = drop_layer(document_representation) ## [num_labels, batch_size, hidden_size * 4]
drop_step_reshape = drop_step.permute(1,2,0)## [batch_size, hidden_size * 4, num_labels]
logits = drop_step_reshape * W_projection.T## [batch_size, hidden_size * 4, num_labels]
logits = logits.sum(dim = 1)## [batch_size, num_labels]